In [1]:
import os
import math
import re
from collections import defaultdict


STOP_WORDS = {
    "a", "an", "the", "they", "these", "this", "for", "is", "are",
    "was", "of", "or", "and", "does", "will", "whose"
}

PUNCTUATION_REGEX = r"[{}\[\]<>=(\).,;'\"?!#\-:]"

def normalize_word(word):
    word = word.lower()
    
    # basic singularization (as per assignment examples)
    if word.endswith("s"):
        word = word[:-1]
    
    return word

class Position:
    def __init__(self, page, index):
        self.page = page
        self.index = index
    
    def getPageEntry(self):
        return self.page
    
    def getWordIndex(self):
        return self.index

class WordEntry:
    def __init__(self, word):
        self.word = word
        self.positions = []
    
    def addPosition(self, position):
        self.positions.append(position)
    
    def addPositions(self, positions):
        self.positions.extend(positions)
    
    def getAllPositionsForThisWord(self):
        return self.positions
    
    def getTermFrequency(self, page):
        count = 0
        total_words = page.total_words
        
        for pos in self.positions:
            if pos.getPageEntry() == page:
                count += 1
        
        if total_words == 0:
            return 0
        
        return count / total_words

class PageIndex:
    def __init__(self):
        self.word_entries = {}
    
    def addPositionForWord(self, word, position):
        if word not in self.word_entries:
            self.word_entries[word] = WordEntry(word)
        
        self.word_entries[word].addPosition(position)
    
    def getWordEntries(self):
        return list(self.word_entries.values())

class PageEntry:
    def __init__(self, pageName, base_path):
        self.pageName = pageName
        self.pageIndex = PageIndex()
        self.total_words = 0
        
        file_path = os.path.join(base_path, pageName)
        
        with open(file_path, 'r') as f:
            text = f.read()
        
        # remove punctuation
        text = re.sub(PUNCTUATION_REGEX, " ", text)
        
        words = text.split()
        
        index = 1
        
        for word in words:
            clean_word = normalize_word(word)
            
            if clean_word not in STOP_WORDS:
                pos = Position(self, index)
                self.pageIndex.addPositionForWord(clean_word, pos)
            
            index += 1
        
        self.total_words = index - 1
    
    def getPageIndex(self):
        return self.pageIndex

class MyHashTable:
    def __init__(self):
        self.table = defaultdict(list)
    
    def addPositionsForWord(self, wordEntry):
        word = wordEntry.word
        
        existing_entries = self.table[word]
        
        if not existing_entries:
            self.table[word].append(wordEntry)
        else:
            existing_entries[0].addPositions(wordEntry.getAllPositionsForThisWord())

class InvertedPageIndex:
    def __init__(self):
        self.hashTable = MyHashTable()
        self.pages = []
    
    def addPage(self, page):
        self.pages.append(page)
        
        for wordEntry in page.getPageIndex().getWordEntries():
            self.hashTable.addPositionsForWord(wordEntry)
    
    def getPagesWhichContainWord(self, word):
        word = normalize_word(word)
        
        if word not in self.hashTable.table:
            return []
        
        positions = self.hashTable.table[word][0].getAllPositionsForThisWord()
        
        pages = set()
        for pos in positions:
            pages.add(pos.getPageEntry().pageName)
        
        return list(pages)
    
    def getPositions(self, word, pageName):
        word = normalize_word(word)
        
        if word not in self.hashTable.table:
            return []
        
        positions = self.hashTable.table[word][0].getAllPositionsForThisWord()
        
        result = []
        for pos in positions:
            if pos.getPageEntry().pageName == pageName:
                result.append(pos.getWordIndex())
        
        return result


class SearchEngine:
    def __init__(self, base_path):
        self.index = InvertedPageIndex()
        self.base_path = base_path
        self.loaded_pages = set()
    
    def performAction(self, action):
        parts = action.strip().split()
        
        if parts[0] == "addPage":
            page = parts[1]
            
            if page not in self.loaded_pages:
                pe = PageEntry(page, self.base_path)
                self.index.addPage(pe)
                self.loaded_pages.add(page)
        
        elif parts[0] == "queryFindPagesWhichContainWord":
            word = parts[1]
            pages = self.index.getPagesWhichContainWord(word)
            
            if not pages:
                print(f"No webpage contains word {word}")
            else:
                print(", ".join(sorted(pages)))
        
        elif parts[0] == "queryFindPositionsOfWordInAPage":
            word = parts[1]
            page = parts[2]
            
            if page not in self.loaded_pages:
                print(f"No webpage {page} found")
                return
            
            positions = self.index.getPositions(word, page)
            
            if not positions:
                print(f"Webpage {page} does not contain word {word}")
            else:
                print(", ".join(map(str, sorted(positions))))

base_path = "../datasets/Q2- webSearch/webpages"

engine = SearchEngine(base_path)

actions_file = "../datasets/Q2- webSearch/actions.txt"

with open(actions_file, "r") as f:
    actions = f.readlines()

print("=========== OUTPUT ===========")

for action in actions:
    print(f"\n>> {action.strip()}")
    engine.performAction(action)

=========== OUTPUT ===========

>> addPage stack_datastructure_wiki

>> queryFindPagesWhichContainWord delhi
No webpage contains word delhi

>> queryFindPagesWhichContainWord stack
stack_datastructure_wiki

>> queryFindPagesWhichContainWord wikipedia
stack_datastructure_wiki

>> queryFindPositionsOfWordInAPage magazines stack_datastructure_wiki
Webpage stack_datastructure_wiki does not contain word magazines

>> queryFindPagesWhichContainWord allain
No webpage contains word allain

>> addPage stack_cprogramming

>> queryFindPagesWhichContainWord allain
stack_cprogramming

>> queryFindPagesWhichContainWord C
stack_cprogramming

>> queryFindPagesWhichContainWord C++
stack_cprogramming

>> addPage stack_oracle

>> queryFindPagesWhichContainWord jdk
stack_oracle

>> addPage stackoverflow

>> queryFindPagesWhichContainWord function
stack_cprogramming, stack_datastructure_wiki, stackoverflow

>> addPage stacklighting

>> addPage stackmagazine

>> queryFindPagesWhichContainWord magazines
stac